In [0]:
from pyspark.sql import functions as F

cols = ["sacc_code", "country", "geography", "financial_year", "value", "_measure"]
mig = (spark.table("workspace.migration.bronze_net").select(cols)
       .unionByName(spark.table("workspace.migration.bronze_arrivals").select(cols))
       .unionByName(spark.table("workspace.migration.bronze_departures").select(cols)))

silver = (mig
    .withColumnRenamed("_measure", "measure")
    .withColumn("value", F.col("value").cast("int"))
    .withColumn("sacc_code", F.col("sacc_code").cast("int"))
    .withColumn("year_start", F.substring("financial_year", 1, 4).cast("int"))
    .withColumn("is_preliminary", F.col("financial_year") == F.lit("2024-25"))
    .withColumn("is_national_total",
                (F.col("geography") == "Australia") & (F.col("country") == "Australia")))

silver.write.mode("overwrite").saveAsTable("workspace.migration.silver_migration")

In [0]:
visa = (spark.table("workspace.migration.bronze_visa")
        .withColumn("value", F.col("value").cast("int"))
        .withColumn("year_start", F.substring("financial_year", 1, 4).cast("int")))

visa.write.mode("overwrite").saveAsTable("workspace.migration.silver_visa")

In [0]:
%sql
SELECT financial_year,
       SUM(CASE WHEN measure='net' THEN value END) AS net,
       SUM(CASE WHEN measure='arrivals' THEN value END)
         - SUM(CASE WHEN measure='departures' THEN value END) AS arrivals_minus_departures
FROM workspace.migration.silver_migration
WHERE geography = 'Australia' AND NOT is_national_total
GROUP BY financial_year ORDER BY financial_year;

financial_year,net,arrivals_minus_departures
2004-05,161300,161200
2005-06,186520,186560
2006-07,245640,245720
2007-08,292660,292620
2008-09,302240,302190
2009-10,199810,199680
2010-11,188580,188590
2011-12,240060,239960
2012-13,240220,240240
2013-14,207150,207160
